# 00 — Dataset Preparation

Loads HIGGS.csv, creates 3 imbalance versions, saves all train/test splits to `data/`.

**⚠️ Run this FIRST before any experiment notebook.**

**Kernel:** Select `higgs_venv` in VS Code top-right before running.

**SAMPLE_MODE:** Default = True (500k rows, ~2 min). Set False for final overnight paper run (~15 min, needs ~8GB RAM).

In [1]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath('..'))
from utils.data_loader import load_raw, create_versions, split_version

# ─────────────────────────────────────────────────────────────────────────────
# FLIP TO False FOR FINAL PAPER RUN ONLY
# True  → 500k rows  → ~2 min  → safe on 16GB RAM
# False → 11M rows   → ~15 min → needs ~8GB RAM free → run overnight
# ─────────────────────────────────────────────────────────────────────────────
SAMPLE_MODE = True

DATA_DIR = os.path.join('..', 'data')
os.makedirs(DATA_DIR, exist_ok=True)
print(f"SAMPLE_MODE = {SAMPLE_MODE}")
print(f"Data directory: {os.path.abspath(DATA_DIR)}")

SAMPLE_MODE = True
Data directory: D:\higgs_Research\data


In [2]:
df       = load_raw(sample_mode=SAMPLE_MODE)
versions = create_versions(df)

[data_loader] Loading HIGGS.csv — SAMPLE MODE (500,000 rows)


[data_loader] Loaded: 500,000 rows | Signal=264,703 | Background=235,297 | Ratio=1:0.9


[data_loader] Version A: 500,000 rows | Signal=264,703 | Background=235,297 | Ratio=1:0.9
[data_loader] Version B: 258,826 rows | Signal=23,529 | Background=235,297 | Ratio=1:10.0
[data_loader] Version C: 240,002 rows | Signal=4,705 | Background=235,297 | Ratio=1:50.0


In [3]:
feature_cols = [f'feature_{i}' for i in range(1, 29)]

for v_name, v_df in versions.items():
    X_train, X_test, y_train, y_test = split_version(v_df, v_name)

    train_df = pd.DataFrame(X_train, columns=feature_cols)
    train_df.insert(0, 'label', y_train)
    test_df  = pd.DataFrame(X_test, columns=feature_cols)
    test_df.insert(0, 'label', y_test)

    train_path = os.path.join(DATA_DIR, f'version_{v_name}_train.csv')
    test_path  = os.path.join(DATA_DIR, f'version_{v_name}_test.csv')

    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path,  index=False)

    print(f"Version {v_name} → train: {len(train_df):,} rows ({os.path.getsize(train_path)/1e6:.1f}MB) | "
          f"test: {len(test_df):,} rows ({os.path.getsize(test_path)/1e6:.1f}MB)")

print("\n✓ All splits saved.")

[data_loader] Version A split → Train: 400,000 | Test: 100,000 | Train signal ratio: 0.529


Version A → train: 400,000 rows (202.1MB) | test: 100,000 rows (50.5MB)
[data_loader] Version B split → Train: 207,060 | Test: 51,766 | Train signal ratio: 0.091


Version B → train: 207,060 rows (104.7MB) | test: 51,766 rows (26.2MB)
[data_loader] Version C split → Train: 192,001 | Test: 48,001 | Train signal ratio: 0.020


Version C → train: 192,001 rows (97.1MB) | test: 48,001 rows (24.3MB)

✓ All splits saved.


In [4]:
print("Verifying saved files...\n")
for v in ['A', 'B', 'C']:
    train = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_train.csv'))
    test  = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_test.csv'))
    sig   = train['label'].sum()
    bg    = (train['label'] == 0).sum()
    print(f"Version {v} — Train: {len(train):,} | Signal={sig:,} | Background={bg:,} | Ratio=1:{bg/sig:.1f}")
    print(f"         Test:  {len(test):,} rows")
print("\n✓ Verification complete. Ready to run experiment notebooks.")

Verifying saved files...



Version A — Train: 400,000 | Signal=211,762 | Background=188,238 | Ratio=1:0.9
         Test:  100,000 rows


Version B — Train: 207,060 | Signal=18,823 | Background=188,237 | Ratio=1:10.0
         Test:  51,766 rows


Version C — Train: 192,001 | Signal=3,764 | Background=188,237 | Ratio=1:50.0
         Test:  48,001 rows

✓ Verification complete. Ready to run experiment notebooks.
